In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import pylab
from PIL import Image
from math import e
import astropy.io.fits as pf
from astropy.io import fits
from mpl_toolkits.axes_grid1 import make_axes_locatable
from tqdm import tqdm
from astropy.convolution import convolve
from astropy.convolution import Gaussian2DKernel

In [ ]:
CGPS = fits.open('/srv/data/cgps-gmims/cgps_for_sims/all_stokesi_allbands9.fits')
hdr1 = CGPS[0].header
data_ST = CGPS[0].data#[0,0]
print(data_ST.shape)
print(hdr1['CDELT1'])
print('')

CHIME = fits.open('/srv/data/chime/chime_QU_Oct2023_400_729/I_400_729_Oct2023_new.fits')
#CHIME = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2024_new.fits')
hdr2 = CHIME[0].header
data_CHIME = CHIME[0].data[1]
print(data_CHIME.shape)
print(hdr2['CDELT1'])
print(repr(hdr2))

dxy_CGPS  =  np.round(hdr1['CDELT2'],5)
dxy_CHIME =  np.round(hdr2['CDELT2'],5)

print(dxy_CGPS,dxy_CHIME)

data_CHIME[np.isnan(data_CHIME)] = 0.0

In [ ]:
def get_uv_axis(image, dxy, freqMHz=1420, method='right'):

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy

    print(uv_freq_m)
    print(uv_freq_n)

    if method == 'right':
        #uv_m = 2*uv_freq_m*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
        #uv_n = 2*uv_freq_n*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
        uv_m = 2*uv_freq_m*(180/np.pi)*(3e8/(freqMHz*1e6))
        uv_n = 2*uv_freq_n*(180/np.pi)*(3e8/(freqMHz*1e6))

    if method == 'wrong':
        uv_m = uv_freq_m*(3e8/(freqMHz*1e6))*180/(np.pi**2)
        uv_n = uv_freq_n*(3e8/(freqMHz*1e6))*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_n, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    uv = {}
    uv['uvm'] = uv_m
    uv['uvn'] = uv_n
    uv['xuv'] = xuv
    uv['yuv'] = yuv
    uv['ruv'] = ruv
    
    return uv

In [ ]:
def get_uvbounds_idx(uv,maxu,maxv):

    du = uv['uvm'][1] - uv['uvm'][0]
    dv = uv['uvn'][1] - uv['uvn'][0]

    print(du,dv)

    idx_u1 = abs(uv['uvm'] + maxu).argmin()
    idx_u2 = abs(uv['uvm'] - maxu).argmin()
    idx_v1 = abs(uv['uvn'] + maxv).argmin()
    idx_v2 = abs(uv['uvn'] - maxv).argmin()

    print(uv['uvm'][idx_u1],uv['uvm'][idx_u2])
    print(uv['uvn'][idx_v1],uv['uvn'][idx_v2])

    idx = {}
    idx['u1'] = idx_u1
    idx['u2'] = idx_u2
    idx['v1'] = idx_v1
    idx['v2'] = idx_v2

    extent = [uv['uvm'][idx_u1]-du/2, uv['uvm'][idx_u2]+du/2,
              uv['uvn'][idx_v1]-dv/2, uv['uvn'][idx_v2]+dv/2]



    return idx, extent

In [ ]:
def regular_image(data,j0,i0,nxy,dxy,taper=False,taper_width=50):

    image = data[j0:j0+nxy,i0:i0+nxy]

    pix0 = int((nxy-1)/2)
    widxy = nxy*dxy
    
    if taper:      
        x, y = np.meshgrid(np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy), 
                       np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy))
        r = np.sqrt(x**2+y**2)
        
        taper = gf(nxy/2-taper_width,taper_width,r)
        taper[np.where(r<=nxy/2-taper_width)] = 1
        image = image*taper

    print('Width of each pixel: '+str(dxy)+ ' deg.')
    print('Number of x and y pixels: '+str(nxy))
    print('Central pixel index: '+str(pix0))
    print('Width of image: '+str(widxy)+' deg.')
    
    return image,dxy,pix0,widxy

In [ ]:
image_ST_true,dxy,pix0,widxy = regular_image(data_ST,400,22555,1023,dxy_CGPS)

fig,ax = plt.subplots(1,figsize=(6,6)) 
ax.imshow(image_ST_true,origin='lower',vmin=0,vmax=0.2)
ax.set_title('ST only (actual)')


In [ ]:
image_FFT = np.fft.fft2(image_ST_true)
image_FFT_shift = np.fft.fftshift(image_FFT)

# Generate uv-plane axes and Gaussian beam low-pass mask:
uv = get_uv_axis(image_ST_true, dxy)
idx, extent = get_uvbounds_idx(uv,1800,1800)

fig,ax = plt.subplots(1,2,figsize=(16,8)) 
ax[0].imshow(abs(image_FFT_shift)[idx['v1']:idx['v2']+1,idx['u1']:idx['u2']+1],
          origin='lower',vmin=0,vmax=50,extent=extent)#,norm='log')
ax[0].grid()
ax[0].set_xlabel('u (m)')
ax[0].set_ylabel('v (m)')

ax[1].plot(uv['uvm'], abs(image_FFT_shift)[511])
ax[1].set_yscale('log')
ax[1].set_xlim(-1800,1800)
ax[1].set_ylim(1,1e5)
ax[1].grid()
ax[1].set_xlabel('u (m)')

plt.savefig('../plots/st_uv_wfactor2.png')

In [ ]:
image_FFT = np.fft.fft2(data_CHIME)
image_FFT_shift = np.fft.fftshift(image_FFT)

# Generate uv-plane axes and Gaussian beam low-pass mask:
uv = get_uv_axis(data_CHIME, dxy_CHIME, freqMHz=729)
idx, extent = get_uvbounds_idx(uv,100,100)

fig,ax = plt.subplots(1,2,figsize=(16,8)) 
ax[0].imshow(abs(image_FFT_shift)[idx['v1']:idx['v2']+1,idx['u1']:idx['u2']+1],
          origin='lower',vmin=0,vmax=5e4,extent=extent)#,norm='log')
ax[0].axvline(x=22,color='red')
ax[0].axvline(x=44,color='red')
ax[0].axvline(x=66,color='red')
ax[0].grid(None)
ax[0].set_xlabel('u (m)')
ax[0].set_ylabel('v (m)')

#print(uv['uvm'].shape,abs(image_FFT_shift).shape)

ax[1].plot(uv['uvm'], abs(image_FFT_shift)[720])
ax[1].set_yscale('log')
ax[1].set_xlim(-100,100)
ax[1].set_ylim(1,5e5)
ax[1].grid()
ax[1].axvline(x=22,color='red')
ax[1].axvline(x=44,color='red')
ax[1].axvline(x=66,color='red')
ax[1].set_xlabel('u (m)')

plt.savefig('../plots/chime_uv_better.png')

In [ ]:
print(data_CHIME.shape)

In [ ]:
print(repr(hdr2))